# Notebook 3 — Comparação Experimental de Estratégias de Generalização

**Objetivo didático:** comparar intervenções de regularização com base em evidências experimentais.

## Estratégias a comparar
1. Sem regularização
2. Apenas L2
3. Apenas Dropout
4. L2 + Dropout

## Competência central
Você deve sair deste notebook sabendo responder:
**"qual estratégia faz mais sentido neste problema e por quê?"**

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Verificar se a GPU está disponível
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f'Usando o dispositivo: {device}')

In [ ]:
X, y = make_moons(n_samples=800, noise=0.30, random_state=SEED)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=128, shuffle=False)

plt.figure(figsize=(6, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, s=12)
plt.title("Conjunto de treino (padronizado)")
plt.show()

In [ ]:
class ExperimentMLP(nn.Module):
    def __init__(self, input_dim, dropout=0.0):
        super().__init__()
        layers = [
            nn.Linear(input_dim, 64),
            nn.ReLU(),
        ]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        layers += [
            nn.Linear(64, 64),
            nn.ReLU(),
        ]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        layers += [nn.Linear(64, 2)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total = 0

    with torch.set_grad_enabled(is_train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)

    return total_loss / total, total_correct / total


def fit(model, train_loader, val_loader, criterion, optimizer, epochs=120):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for _ in range(epochs):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader, criterion)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)

    return history


def summarize_history(name, history):
    train_acc = history["train_acc"][-1]
    val_acc = history["val_acc"][-1]
    train_loss = history["train_loss"][-1]
    val_loss = history["val_loss"][-1]
    gap = train_acc - val_acc
    return {
        "configuração": name,
        "train_acc_final": round(train_acc, 4),
        "val_acc_final": round(val_acc, 4),
        "train_loss_final": round(train_loss, 4),
        "val_loss_final": round(val_loss, 4),
        "gap_acc": round(gap, 4),
    }


def plot_many(histories):
    plt.figure(figsize=(8, 5))
    for name, hist in histories.items():
        plt.plot(hist["val_acc"], label=name)
    plt.xlabel("Época")
    plt.ylabel("Validation Accuracy")
    plt.title("Comparação das curvas de validação")
    plt.legend()
    plt.show()

## Executando os cenários padrão

Você pode executar sem alterações para obter um ponto de partida.  
Depois, altere:
- o valor de `weight_decay`
- o valor de `dropout`
- o número de épocas
- a largura da rede

O importante é **justificar** a sua escolha final.

In [ ]:
criterion = nn.CrossEntropyLoss()

configs = {
    "baseline": {"dropout": 0.0, "weight_decay": 0.0},
    "l2": {"dropout": 0.0, "weight_decay": 1e-3},
    "dropout": {"dropout": 0.3, "weight_decay": 0.0},
    "l2+dropout": {"dropout": 0.3, "weight_decay": 1e-3},
}

histories = {}
rows = []

for name, cfg in configs.items():
    torch.manual_seed(SEED)
    model = ExperimentMLP(input_dim=2, dropout=cfg["dropout"]).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=cfg["weight_decay"]
    )

    history = fit(model, train_loader, val_loader, criterion, optimizer, epochs=120)
    histories[name] = history
    rows.append(summarize_history(name, history))

results_df = pd.DataFrame(rows).sort_values("val_acc_final", ascending=False)
results_df

In [ ]:
plot_many(histories)

## Análise obrigatória

Responda com base na tabela e nas curvas:
1.	Qual configuração superou a baseline de forma mais convincente?
2.	O ganho em validação veio acompanhado de redução do gap ou apenas de aumento de capacidade?
3.  Houve caso em que a regularização reduziu demais a capacidade do modelo?
4.	A combinação L2 + Dropout trouxe benefício adicional real ou regularizou em excesso?
5.	Entre L2 e Dropout, qual intervenção isolada foi mais eficaz neste cenário?
6.	Essa decisão mudaria se o dataset fosse maior ou mais ruidoso?

## Extensão opcional
Execute novos cenários alterando:
- `dropout = 0.1`, `0.2`, `0.5`
- `weight_decay = 1e-4`, `1e-2`

In [ ]:
# ÁREA LIVRE PARA NOVOS EXPERIMENTOS

extra_configs = {
    "dropout_0.1": {"dropout": 0.1, "weight_decay": 0.0},
    "dropout_0.5": {"dropout": 0.5, "weight_decay": 0.0},
    "l2_1e-4": {"dropout": 0.0, "weight_decay": 1e-4},
}

extra_rows = []

for name, cfg in extra_configs.items():
    torch.manual_seed(SEED)
    model = ExperimentMLP(input_dim=2, dropout=cfg["dropout"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=cfg["weight_decay"])
    history = fit(model, train_loader, val_loader, criterion, optimizer, epochs=120)
    extra_rows.append(summarize_history(name, history))

pd.DataFrame(extra_rows).sort_values("val_acc_final", ascending=False)

## Síntese final

Escreva uma conclusão curta contendo:
- problema identificado;
- estratégia escolhida;
- evidência usada para sustentar a decisão;
- limitação do experimento.